# CapCutPro render worker (Colab, free CPU)

Runtime → **Run all**. The worker listens on port **8787**. When the `WORKER URL` box appears, paste it into the site as `NEXT_PUBLIC_WORKER_URL`. Keep this tab open.

No API key is required. The cell below installs **Ollama** and pulls `qwen2.5:3b` (about 2 GB). If that pull fails, the worker still renders and says that the rule engine did the edit, not a language model.

Free Colab is a shared CPU with roughly 12 GB RAM and a session that can be shut down. Do not expect a vision-language model and Whisper and a 4K60 render at the same time. The default file is 1080×1920 H.264 so a phone can play it. Set `SHORT_DELIVERY=4k` only if you can wait; that file is labeled upscaled when the source is smaller.

Daily YouTube trending files need a free Kaggle token (`KAGGLE_USERNAME` and `KAGGLE_KEY`). Without it the worker uses a bundled 2019–2020 title mirror and says so. No paid API is called.


In [ ]:
import os, shutil, subprocess
shutil.rmtree('/content/capcutpro-agent', ignore_errors=True)
subprocess.run(
    'git clone -q --depth 1 -b cursor/creative-engine-c4cd https://github.com/babyfam777-maker/capcutpro-agent /content/capcutpro-agent',
    shell=True, check=True,
)
print(os.listdir('/content/capcutpro-agent/worker'))


In [ ]:
%%bash
set -e
apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg fonts-noto-core fonts-noto-color-emoji curl libegl1 libgl1 >/dev/null
cd /content/capcutpro-agent/worker
grep -v '^pytest' requirements.txt > /tmp/req.txt
pip -q install -r /tmp/req.txt
pip -q install mediapipe || echo 'MediaPipe did not install; faces fall back to OpenCV Haar and hands are not tracked.'
echo DEPS_OK


In [ ]:
import os, subprocess, time, urllib.request
# Local Ollama. No key. If the pull fails, leave the worker up; edits then say they used the rule engine.
if not os.path.exists('/usr/local/bin/ollama'):
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=False)
serve = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(3)
pulled = subprocess.run(['ollama', 'pull', 'qwen2.5:3b'])
print('OLLAMA_PULL', pulled.returncode)


In [ ]:
import os, subprocess, time, urllib.request
os.chdir('/content/capcutpro-agent/worker')
env = dict(
    os.environ,
    DATA_DIR='/content/data',
    WHISPER_MODEL='base',
    OPENAI_BASE_URL='http://127.0.0.1:11434/v1',
    OPENAI_API_KEY='ollama',
    MODEL='qwen2.5:3b',
    SHORT_DELIVERY='phone',
)
worker = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8787'],
    env=env, stdout=open('/content/worker.log','w'), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8787/health').read().decode())
        break
    except Exception:
        time.sleep(2)
else:
    print(open('/content/worker.log').read())
    raise SystemExit('worker did not start')


In [ ]:
import os, re, subprocess, time, urllib.request
subprocess.run(['wget','-q','-O','/content/cloudflared','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
os.chmod('/content/cloudflared', 0o755)
tunnel = subprocess.Popen(
    ['/content/cloudflared','tunnel','--no-autoupdate','--url','http://localhost:8787'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break
print('=' * 60)
print('WORKER URL:', url)
print('=' * 60)
for _ in range(30):
    try:
        print(urllib.request.urlopen(url + '/health').read().decode())
        break
    except Exception:
        time.sleep(3)


In [ ]:
# Keep alive. Stop this cell to shut the tunnel down.
import threading, time
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
while True:
    print(time.strftime('%H:%M:%S'), 'worker', 'up' if worker.poll() is None else 'DOWN', '| tunnel', 'up' if tunnel.poll() is None else 'DOWN', '|', url, flush=True)
    time.sleep(60)
